In [1]:
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('GK_Questions.pdf')
pages = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
spilitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 30)
chunk_fun = spilitter.split_documents(pages)
chunk = [i.page_content for i in chunk_fun]
chunk[0]

"General Knowledge — Q&A; Compendium\n310 questions across Science, Politics, History, Geography, Space, Magic & Mythology, AI, and Tech.\nScience\n1. What is the chemical symbol for gold?\nAns: Au\n2. What is the powerhouse of the cell?\nAns: Mitochondria\n3. What gas do plants absorb from the atmosphere for photosynthesis?\nAns: Carbon dioxide\n4. What is the SI unit of electric current?\nAns: Ampere\n5. Who proposed the theory of general relativity?\nAns: Albert Einstein\n6. What is the hardest natural substance on Earth?\nAns: Diamond\n7. What is the boiling point of water at sea level in Celsius?\nAns: 100°C\n8. What particle has a negative charge?\nAns: Electron\n9. What is the study of fungi called?\nAns: Mycology\n10. What is the most abundant gas in Earth's atmosphere?\nAns: Nitrogen\n11. What organ produces insulin?\nAns: Pancreas\n12. What is the chemical formula for table salt?\nAns: NaCl\n13. What force keeps planets in orbit around the Sun?\nAns: Gravity\n14. What is the 

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
embedding = DefaultEmbeddingFunction()

client = chromadb.PersistentClient(path='./agentic_rag_2')
collection = client.get_or_create_collection(name='General_Data', embedding_function=embedding)

if collection.count() == 0:
    collection.add(
        documents=chunk,
        ids=[str(i) for i in range(len(chunk))],
        metadatas=[{'source':'GK_Questions.pdf','chunk_id':i}
                   for i in range(len(chunk))]
    )
collection

Collection(name=General_Data)

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-120b')
from langchain_core.tools import tool
import ast

@tool
def first_fun(query:str):
    '''provide answer based on the context no exaternal cntext'''
    result = collection.query(query_texts=[query], n_results=8)
    distance = result['distances'][0]
    document = result['documents'][0]
    threshold = 1.0
    good_chunks = []
    for doc, dist in zip(document,distance):
        if dist < threshold:
            good_chunks.append(doc)
        if not good_chunks:
            return 'That not from my context'
        return '/n/n'.join(good_chunks)
llm_tool = llm.bind_tools([first_fun])

def tool_fun(question:str):
    massage = [{
        'role':'user',
        'content':question
    }]
    result = llm_tool.invoke(massage)

    if not result.tool_calls:
        return result.content
    massage.append(result)

    for call in result.tool_calls:
        respond = first_fun.invoke(call['args'])
        massage.append(
            {
                'role':'tool',
                'content':'respond',
                'tool_call_id':call['id']
            }
        )
    final_result = llm.invoke(massage)
    return final_result.content
bunch_question = [
    "What is 'overfitting' in machine learning?",
    "What does 'HTML' stand for?",
    "What does 'IoT' stand for?"
]   
for q in bunch_question:
    ans = tool_fun(q)
    print(ans) 


**Overfitting** is a common problem in machine learning (ML) where a model learns the training data **too well**, capturing not only the underlying patterns that generalize to new data but also the random noise and idiosyncrasies specific to that particular dataset. As a result, the model performs **excellent on the training set** but **poorly on unseen (validation or test) data**.

---

## Why Overfitting Happens

| Reason | Explanation |
|--------|-------------|
| **Model Complexity** | Very flexible models (e.g., deep neural networks with many layers, high‑degree polynomial regression, decision trees with many leaves) have enough capacity to memorize the training points. |
| **Insufficient Data** | When the training set is small relative to the model’s capacity, the model can “memorize” each example instead of learning general trends. |
| **Noisy Labels / Features** | Errors or random fluctuations in the data become part of the learned function if the model tries to fit them. |
| **